# Cuaderno 1 — Verificación del Entorno de Desarrollo

Este cuaderno comprueba que todas las bibliotecas necesarias para la guía
_Inteligencia Artificial en Ciberseguridad_ están correctamente instaladas
y disponibles en el entorno Python activo.

**Bibliotecas cubiertas:**
| Biblioteca | Uso principal |
|---|---|
| `numpy` | Operaciones matriciales |
| `pandas` | Manipulación de datos tabulares |
| `scikit-learn` | Algoritmos de ML clásicos |
| `matplotlib` / `seaborn` | Visualización |
| `imbalanced-learn` | Manejo de clases desbalanceadas (SMOTE) |
| `shap` | Explicabilidad de modelos (XAI) |
| `pefile` | Análisis de ejecutables PE |
| `joblib` | Serialización de modelos |

In [ ]:
import sys
import importlib

# Lista de paquetes requeridos
REQUIRED = [
    "numpy", "pandas", "sklearn", "matplotlib",
    "seaborn", "imblearn", "shap", "pefile", "joblib"
]

print(f"Python {sys.version}")
print("=" * 50)
all_ok = True
for pkg in REQUIRED:
    try:
        mod = importlib.import_module(pkg)
        version = getattr(mod, "__version__", "disponible")
        print(f"  [OK]    {pkg:<20} {version}")
    except ImportError:
        print(f"  [FALTA] {pkg}")
        all_ok = False

print("=" * 50)
print("Estado final:", "✓ Todo listo" if all_ok else "✗ Hay paquetes faltantes")

## Prueba funcional básica

Verificamos que cada biblioteca opera correctamente ejecutando una operación
representativa de lo que se usa en los cuadernos siguientes.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import seaborn as sns

# ── Numpy: operación matricial ──────────────────────────────────────────
arr = np.random.rand(100, 4)
print(f"[numpy]     array shape={arr.shape}, mean={arr.mean():.4f}")

# ── Pandas: creación de DataFrame ──────────────────────────────────────
df = pd.DataFrame(arr, columns=["A", "B", "C", "D"])
print(f"[pandas]    DataFrame {df.shape}")

# ── scikit-learn: escalado + modelo ────────────────────────────────────
scaler = MinMaxScaler()
scaled = scaler.fit_transform(arr)
model = IsolationForest(random_state=42)
model.fit(scaled)
print(f"[sklearn]   IsolationForest entrenado, n_estimators={model.n_estimators}")

# ── Matplotlib / Seaborn: gráfico de prueba ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].hist(arr[:, 0], bins=20, color="steelblue", edgecolor="white")
axes[0].set_title("Histograma (matplotlib)")
sns.kdeplot(arr[:, 1], ax=axes[1], fill=True, color="darkorange")
axes[1].set_title("KDE (seaborn)")
plt.suptitle("Prueba de visualización", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()
print("[matplotlib/seaborn] OK")

In [ ]:
# ── imbalanced-learn: SMOTE ─────────────────────────────────────────────
from imblearn.over_sampling import SMOTE
X_imb = np.vstack([np.random.randn(90, 4), np.random.randn(10, 4) + 5])
y_imb = np.array([0] * 90 + [1] * 10)
sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X_imb, y_imb)
print(f"[imblearn]  SMOTE: {dict(zip(*np.unique(y_imb, return_counts=True)))} → "
      f"{dict(zip(*np.unique(y_res, return_counts=True)))}")

# ── shap ────────────────────────────────────────────────────────────────
import shap
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=10, random_state=42)
rf.fit(X_res, y_res)
explainer = shap.TreeExplainer(rf)
svals = explainer.shap_values(X_res[:5])
print(f"[shap]      valores SHAP calculados, shape={np.array(svals).shape}")

# ── joblib ──────────────────────────────────────────────────────────────
import joblib, os
joblib.dump(rf, "/tmp/rf_test.pkl")
rf2 = joblib.load("/tmp/rf_test.pkl")
os.remove("/tmp/rf_test.pkl")
print(f"[joblib]    serialización/carga OK")

print("\n✓ Todas las bibliotecas funcionan correctamente.")